# Workshop Setup

Run this notebook **once** at the start of the workshop to configure shared resource values.

After running, all other workshop notebooks (`1_basic/`, `2_strands/`, `3_marengo_embedding_strategy/`, `4_vector_db_test/`) will automatically load the configuration from `workshop_config.json`.

You will need 3 values from the **CloudFormation Outputs** tab:
- `WorkshopS3BucketName` → S3 Bucket Name
- `WorkshopS3VectorBucketName` → S3 Vector Bucket Name
- `OpenSearchVectorDBEndpoint` → OpenSearch Endpoint

## 1. Enter CloudFormation Output Values

Replace the placeholder values below with your actual CloudFormation outputs.

In [ ]:
# ==============================
# TODO: Replace with your CloudFormation output values
# ==============================
S3_BUCKET_NAME = "<YOUR_S3_BUCKET_NAME>"                    # WorkshopS3BucketName
S3_VECTOR_BUCKET_NAME = "<YOUR_S3_VECTOR_BUCKET_NAME>"      # WorkshopS3VectorBucketName
OPENSEARCH_ENDPOINT = "<YOUR_OPENSEARCH_ENDPOINT>"           # OpenSearchVectorDBEndpoint

## 2. Validate and Save Configuration

In [ ]:
import sys, os
import boto3

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))
from workshop_config import save_config

# --- Validate inputs ---
errors = []

# S3 Bucket (required)
if not S3_BUCKET_NAME or S3_BUCKET_NAME.startswith("<"):
    errors.append("S3_BUCKET_NAME is required. Enter the WorkshopS3BucketName from CloudFormation outputs.")

# S3 Vector Bucket (optional but validated if provided)
if S3_VECTOR_BUCKET_NAME.startswith("<"):
    S3_VECTOR_BUCKET_NAME = ""
    print("Note: S3_VECTOR_BUCKET_NAME not set. S3 Vectors features will be unavailable.")

# OpenSearch Endpoint (optional but validated if provided)
if OPENSEARCH_ENDPOINT.startswith("<"):
    OPENSEARCH_ENDPOINT = ""
    print("Note: OPENSEARCH_ENDPOINT not set. OpenSearch features will be unavailable.")
elif OPENSEARCH_ENDPOINT.startswith("https://"):
    OPENSEARCH_ENDPOINT = OPENSEARCH_ENDPOINT.replace("https://", "")

if errors:
    for e in errors:
        print(f"ERROR: {e}")
    raise ValueError("Please fix the errors above and re-run this cell.")

# --- Auto-detect AWS region and account ---
session = boto3.Session()
AWS_REGION = session.region_name
AWS_ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]

print(f"AWS Region: {AWS_REGION}")
print(f"AWS Account ID: {AWS_ACCOUNT_ID}")

# --- Verify S3 bucket access ---
s3_client = session.client("s3")
try:
    s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
    print(f"S3 Bucket: {S3_BUCKET_NAME}")
except Exception as e:
    raise ValueError(f"Cannot access S3 bucket '{S3_BUCKET_NAME}': {e}")

# --- Save configuration ---
config = {
    "S3_BUCKET_NAME": S3_BUCKET_NAME,
    "S3_VECTOR_BUCKET_NAME": S3_VECTOR_BUCKET_NAME,
    "OPENSEARCH_ENDPOINT": OPENSEARCH_ENDPOINT,
    "AWS_REGION": AWS_REGION,
    "AWS_ACCOUNT_ID": AWS_ACCOUNT_ID,
}

save_config(config)
print("\nWorkshop configuration saved successfully!")

## 3. Verify Saved Configuration

In [ ]:
from workshop_config import load_config

saved = load_config()
print("Saved workshop configuration:")
print("=" * 50)
for key, value in saved.items():
    display_value = value if value else "(not set)"
    print(f"  {key}: {display_value}")
print("=" * 50)
print("\nYou can now run any workshop notebook. They will load this configuration automatically.")